# Performance

## Measurement Scope

This page generates tables at book compilation from the document-reported
aggregates in `milestone1/data/reported_results.json`, reviewed on
**9 November 2026**. Hieu's document supplies arXiv Fast Crawl/Recovery time
and compressed checkpoint sizes; Dat's document supplies cache-heavy Semantic
Scholar rerun timings and notebook-process RSS. Original run exports and exact
per-run configurations are pending reconciliation.

The technical documents report Google Colab CPU-only execution. The supplied
arXiv notebooks configure three workers, while the arXiv document describes a
single active connection; the actual benchmark configuration needs
confirmation. These separate measurements do not yet constitute a complete
end-to-end M1 benchmark.

In [ ]:
from pathlib import Path
import sys

# Support execution from the project root or the notebook's own directory.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "myst.yml").is_file() and (p / "milestone1" / "reporting.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from milestone1.reporting import load_results, total, integer, percent, seconds, hms, table

data = load_results()
arxiv = data["arxiv"]["ranges"]
s2 = data["semantic_scholar"]["ranges"]

## arXiv Recorded Crawl and Recovery Time

**Table 1.** Times as printed in Hieu's document. Ratios use its printed total
for each row. Total recorded time sums logged invocation durations and is not
necessarily elapsed calendar time including gaps between Colab sessions.

In [ ]:
rows = []
for r in arxiv:
    t = seconds(r["total_hms"])
    rows.append([f"Range {r['range']}", r["fast_hms"], r["recovery_hms"], r["total_hms"],
                 percent(seconds(r["recovery_hms"]), t, 1), f"{t / r['assigned']:.3f}"])
overall = data["arxiv"]["reported_overall"]
t = seconds(overall["total_hms"])
rows.append(["Reported overall", overall["fast_hms"], overall["recovery_hms"], overall["total_hms"],
             percent(seconds(overall["recovery_hms"]), t, 1), f"{t / total(arxiv, 'assigned'):.3f}"])
table(["Range", "Fast Crawl (h:m:s)", "Recovery (h:m:s)", "Reported total (h:m:s)",
       "Recovery share", "Recorded s / assigned ID"], rows)

The document reports **61:41:45** of recorded invocation time (about 61.70
hours), or 8.884 seconds per assigned ID. Recovery accounts for approximately
52.1%. Range totals vary from 7:39:30 to 18:41:33, with 5.514--13.459 seconds
per assigned ID. Fast Crawl averages approximately 3.653--4.782 seconds per
assigned ID across the five ranges.

Per-ID time is not per-request latency: metadata batching and multiple source
versions change the number of HTTP requests. It cannot independently establish
request-rate compliance. The final error count also cannot quantify recovery
effectiveness without the pre-Recovery failures and attempted recoveries.
Possible effects of temporary failures and waiting remain hypotheses until
logs are reconciled.

No 18.7-hour parallel completion time is claimed: taking the slowest range
does not establish a measured group speedup, and shared access constraints
must still be respected. Runtime ledgers omit older untracked runs and sessions
terminated before a ledger event was written.

### Rounding and arithmetic reconciliation

The source document's displayed times contain small discrepancies. They are
kept visible below rather than silently replacing the reported benchmark.
Recalculate from unrounded ledger seconds before finalizing the table.

In [ ]:
audit = []
for r in arxiv:
    calculated = seconds(r["fast_hms"]) + seconds(r["recovery_hms"])
    reported = seconds(r["total_hms"])
    if calculated != reported:
        audit.append([f"Range {r['range']}: phase sum", r["total_hms"], hms(calculated), f"{calculated - reported:+d}"])
for label, key in (("Overall Fast Crawl: row sum", "fast_hms"),
                   ("Overall Recovery: row sum", "recovery_hms"),
                   ("Overall total: row sum", "total_hms")):
    calculated = sum(seconds(r[key]) for r in arxiv)
    reported = seconds(overall[key])
    audit.append([label, overall[key], hms(calculated), f"{calculated - reported:+d}"])
phase_sum = sum(seconds(r["fast_hms"]) + seconds(r["recovery_hms"]) for r in arxiv)
audit.append(["Overall: sum of all printed phases", overall["total_hms"], hms(phase_sum),
              f"{phase_sum - seconds(overall['total_hms']):+d}"])
table(["Check", "Reported", "Recomputed from printed values", "Difference (s)"], audit)

## Compressed Checkpoint Storage

**Table 2.** Retained checkpoint archive size, including retained sources and
bookkeeping. Average size uses assigned input IDs, not successful versions.

In [ ]:
rows = [[f"Range {r['range']}", f"{r['checkpoint_mib']:,.2f}",
         f"{r['checkpoint_mib'] * 1024 / r['assigned']:.1f}"] for r in arxiv]
size = total(arxiv, "checkpoint_mib")
rows.append(["Overall", f"{size:,.2f}", f"{size * 1024 / total(arxiv, 'assigned'):.1f}"])
table(["Range", "Compressed checkpoints (MiB)", "KiB / assigned ID"], rows)

The reported retained checkpoint total is **1,360.67 MiB**, approximately
1.33 GiB and 55.7 KiB per assigned ID. Range sizes are 237.15--291.45 MiB.
This is final compressed output storage, not peak disk use during execution.
It does not quantify the storage reduction caused by removing figures without
corresponding before/after source sizes and stated compression conventions.

The code records batch RSS, scratch size, discovery time and source-size
fields, but those values were not supplied in the arXiv document. Existing
`metrics.csv` and checkpoint metrics should be checked before deciding whether
additional measurements are needed. Latest-batch metrics can describe Recovery
rather than the initial crawl, and sampling stops before checkpoint packaging.

## Semantic Scholar Cache-Heavy Rerun

**Table 3.** Supplied rerun timings. The column called
`avg_s2_seconds_per_api_attempt` in code is labeled here by its actual
denominator: freshly queried publications. One lookup may include retries.

In [ ]:
rows = []
for r in s2:
    fresh = r["input_papers"] - r["cache_hits"] - r["invalid_id"]
    rows.append([f"Range {r['range']}", f"{r['fetch_seconds']:.2f}", f"{r['cell_seconds']:.2f}",
                 f"{r['reported_mean_fetch_seconds']:.3f}", integer(fresh), integer(r["cache_hits"]),
                 f"{r['average_rss_mib']:.2f}", f"{r['peak_rss_mib']:.2f}"])
table(["Range", "Fetch time (s)", "Cell time (s)", "Reported mean fetch / fresh paper (s)",
       "Fresh lookups", "Cache hits", "Average RSS (MiB)", "Peak RSS (MiB)"], rows)

Across the five reported reruns, **24,811 of 25,000** inputs are cache hits,
leaving **189** fresh valid-publication lookups. The displayed fetch times do
not represent retrieving all 25,000 publications from the API without cache.
Neither 189 nor the number of papers is an internal HTTP-attempt counter.

Range 4 reports 1.35 seconds of fetch time but 1,411.08 seconds of cell time.
The measurements have different scopes; the supplied aggregates do not
establish whether cache I/O, other local processing or another cause explains
the difference. No causal diagnosis or inter-range API-speed ranking is made.

Per-paper fetch timing ends before normalization/cache writes. The cell timer
includes its local processing but excludes earlier checkpoint loading and later
table construction/validation. `track_run` samples notebook-process RSS every
0.5 seconds, including the process's existing memory. Reported peaks range
from 246.29 to 269.21 MiB; they are observed sampled maxima, not a full-system
M1 peak. Range means are not pooled into an unweighted overall RAM average.

Metrics/CSV are overwritten per rerun; unlike the arXiv runtime ledger, they
are not a cumulative history. Zero final error flags do not establish zero
retries or HTTP 429 responses. Corpus size and peak disk were not supplied.

## Pending Measurements and Reconciliation

**Table 4.** Evidence still needed before declaring the performance report
complete. Absence of a supplied metric is not filled with an estimate.

In [ ]:
table(["Measurement", "Current evidence", "Next evidence needed"], [
    ["arXiv time", "Document-reported logged invocations", "Raw runtime ledgers; rounding and configurations"],
    ["arXiv RAM / discovery / source sizes", "Measurement fields exist in code; values not supplied", "Per-batch metrics and their run scope"],
    ["arXiv disk", "Final compressed checkpoint sizes", "Peak full-system storage, including temporary/backup use"],
    ["Recovery effectiveness", "Final 22 publication errors", "Pre-Recovery counts and version/reason history"],
    ["Semantic Scholar time", "Cache-heavy rerun cell/fetch timers", "Original fresh-crawl logs or a clearly scoped benchmark"],
    ["Semantic Scholar retries / disk", "No independent aggregate supplied", "Recorded retry/status counters and storage measurement"],
    ["Full M1 benchmark", "Separate stage measurements", "Auditable CPU-only end-to-end wall time and resource footprint"],
])

The two stages' unrelated run timings are not added to invent an end-to-end
measurement. Once the owners provide the original exports or scoped additional
measurements, this page can be updated without changing the report URL.